# Pruebas de idempotencia

Ejecuta el flujo Silver, DLQ y Gold contra un catálogo sandbox separado. Lee Bronze del catálogo fuente, pero no modifica el catálogo principal. Por defecto valida dos ejecuciones con el mismo corte, duplicados y huellas de contenido. La prueba de mutación queda desactivada y requiere habilitarse explícitamente.

In [0]:
import json
import time
import uuid

dbutils.widgets.text("source_catalog", "steam_game_intelligence")
dbutils.widgets.text("sandbox_catalog", "steam_game_intelligence_sbx")
dbutils.widgets.text("orchestrator_path", "./silver_orquestator")
dbutils.widgets.text("sandbox_suffix", "_sbx")
dbutils.widgets.text("timeout_seconds", "3600")
dbutils.widgets.dropdown("run_mutation_tests", "false", ["false", "true"])
dbutils.widgets.dropdown("cleanup_sandbox", "false", ["false", "true"])
dbutils.widgets.dropdown("reset_sandbox_catalog", "false", ["false", "true"])
source_catalog = dbutils.widgets.get("source_catalog")
sandbox_catalog = dbutils.widgets.get("sandbox_catalog")
orchestrator_path = dbutils.widgets.get("orchestrator_path")
suffix = dbutils.widgets.get("sandbox_suffix")
timeout_seconds = int(dbutils.widgets.get("timeout_seconds"))
run_mutation_tests = dbutils.widgets.get("run_mutation_tests").lower() == "true"
cleanup_sandbox = dbutils.widgets.get("cleanup_sandbox").lower() == "true"
reset_sandbox_catalog = dbutils.widgets.get("reset_sandbox_catalog").lower() == "true"
if source_catalog == sandbox_catalog:
    raise ValueError("source_catalog y sandbox_catalog deben ser distintos para proteger el catálogo principal.")
if not sandbox_catalog.endswith("_sbx"):
    raise ValueError("sandbox_catalog debe terminar en _sbx para evitar borrar un catálogo productivo por accidente.")
run_id = str(uuid.uuid4())
cutoff_us = str(int(time.time() * 1_000_000))
schemas = {
    "bronze": f"bronze{suffix}",
    "silver": f"silver{suffix}",
    "dlq": f"dlq{suffix}",
    "gold": f"gold{suffix}",
    "audit": f"audit{suffix}",
}

def table(schema, name):
    return f"`{sandbox_catalog}`.`{schema}`.`{name}`"
def run_orchestrator(current_run_id):
    args = {
        "catalog": sandbox_catalog,
        "bronze_schema": schemas["bronze"],
        "silver_schema": schemas["silver"],
        "dlq_schema": schemas["dlq"],
        "gold_schema": schemas["gold"],
        "audit_schema": schemas["audit"],
        "cutoff_us": cutoff_us,
        "run_id": current_run_id,
        "run_ingestion": "false",
        "timeout_seconds": str(timeout_seconds),
    }
    result = json.loads(dbutils.notebook.run(orchestrator_path, timeout_seconds, args))
    if result.get("status") != "completed":
        raise RuntimeError(f"El orquestador no terminó correctamente: {result}")
    return result

def assert_condition(test_id, condition, detail):
    status = "PASS" if condition else "FAIL"
    print(f"{status} {test_id}: {detail}")
    if not condition:
        raise AssertionError(f"{test_id}: {detail}")


## Preparación del sandbox

Se copian solo las tablas necesarias. Games y SteamCharts usan una selección determinista; Twitch conserva todas sus filas para probar el cruce completo.

In [0]:
if reset_sandbox_catalog:
    spark.sql(f"DROP CATALOG IF EXISTS `{sandbox_catalog}` CASCADE")
    print(f"Catálogo sandbox reiniciado: {sandbox_catalog}")
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{sandbox_catalog}`")
for schema in schemas.values():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{sandbox_catalog}`.`{schema}`")
spark.sql(
    f"CREATE OR REPLACE TABLE {table(schemas['bronze'], 'steam_games_batch')} AS "
    f"SELECT * FROM `{source_catalog}`.`bronze`.`steam_games_batch` "
    "WHERE try_cast(app_id AS BIGINT) IS NOT NULL "
    "AND pmod(try_cast(app_id AS BIGINT), 20) = 0"
)
spark.sql(
    f"CREATE OR REPLACE TABLE {table(schemas['bronze'], 'steam_game_reviews_batch')} AS "
    f"SELECT * FROM `{source_catalog}`.`bronze`.`steam_game_reviews_batch`"
)
spark.sql(
    f"CREATE OR REPLACE TABLE {table(schemas['bronze'], 'steamcharts_player_counts_monthly')} AS "
    f"SELECT * FROM `{source_catalog}`.`bronze`.`steamcharts_player_counts_monthly` "
    "WHERE try_cast(steam_appid AS BIGINT) IS NOT NULL "
    "AND pmod(try_cast(steam_appid AS BIGINT), 20) = 0"
)
spark.sql(
    f"CREATE OR REPLACE TABLE {table(schemas['bronze'], 'twitch_top_games_monthly')} AS "
    f"SELECT * FROM `{source_catalog}`.`bronze`.`twitch_top_games_monthly`"
)
print(f"Sandbox preparado: {sandbox_catalog}.{schemas['bronze']}")


## T1. Repetición con el mismo corte

Se ejecuta dos veces el mismo orquestador con el mismo `cutoff_us`. Las huellas excluyen metadatos variables de ejecución.

In [0]:
run_orchestrator(f"{run_id}_first")
target_tables = [
    ("silver.steam_games", table(schemas["silver"], "steam_games")),
    ("silver.steam_game_reviews", table(schemas["silver"], "steam_game_reviews")),
    ("silver.steam_player_counts_monthly", table(schemas["silver"], "steam_player_counts_monthly")),
    ("silver.twitch_game_activity_monthly", table(schemas["silver"], "twitch_game_activity_monthly")),
    ("dlq.steam_games_rejected", table(schemas["dlq"], "steam_games_rejected")),
    ("dlq.steam_game_reviews_rejected", table(schemas["dlq"], "steam_game_reviews_rejected")),
    ("dlq.steam_player_counts_monthly_rejected", table(schemas["dlq"], "steam_player_counts_monthly_rejected")),
    ("dlq.twitch_game_activity_monthly_rejected", table(schemas["dlq"], "twitch_game_activity_monthly_rejected")),
    ("gold.game_growth", table(schemas["gold"], "game_growth")),
    ("gold.twitch_steam_relationship", table(schemas["gold"], "twitch_steam_relationship")),
    ("gold.dim_game", table(schemas["gold"], "dim_game")),
]
def fingerprint(table_name, qualified_table):
    columns = [
        field.name for field in spark.table(qualified_table).schema.fields
        if field.name not in {"_run_id", "_rejected_at", "generated_at"}
    ]
    struct_expr = ", ".join(f"`{column}`" for column in columns)
    row = spark.sql(
        f"SELECT CAST(COUNT(*) AS BIGINT) AS row_count, "
        f"SUM(CAST(xxhash64(struct({struct_expr})) AS DECIMAL(38,0))) AS fingerprint "
        f"FROM {qualified_table}"
    ).first()
    return {"table": table_name, "row_count": row["row_count"], "fingerprint": row["fingerprint"]}
before = {name: fingerprint(name, qualified) for name, qualified in target_tables}
run_orchestrator(f"{run_id}_second")
after = {name: fingerprint(name, qualified) for name, qualified in target_tables}
for name in before:
    assert_condition(
        "T1",
        before[name]["row_count"] == after[name]["row_count"]
        and before[name]["fingerprint"] == after[name]["fingerprint"],
        f"{name}: conteo y huella idénticos en ambas ejecuciones",
    )


## T2 y T3. Duplicados en Silver, Gold y DLQ

Las claves se comprueban según el grano documentado. En DLQ se comprueba la unicidad de `_reject_key`.

In [0]:
duplicate_checks = [
    ("silver.steam_games", table(schemas["silver"], "steam_games"), ["app_id"]),
    ("silver.steam_game_reviews", table(schemas["silver"], "steam_game_reviews"), ["app_id"]),
    ("silver.steam_player_counts_monthly", table(schemas["silver"], "steam_player_counts_monthly"), ["app_id", "month_start"]),
    ("silver.twitch_game_activity_monthly", table(schemas["silver"], "twitch_game_activity_monthly"), ["month_start", "twitch_rank"]),
    ("gold.game_growth", table(schemas["gold"], "game_growth"), ["app_id", "month_start"]),
    ("gold.twitch_steam_relationship", table(schemas["gold"], "twitch_steam_relationship"), ["app_id", "month_start"]),
    ("gold.dim_game", table(schemas["gold"], "dim_game"), ["app_id"]),
]
for name, qualified, keys in duplicate_checks:
    key_sql = ", ".join(f"`{key}`" for key in keys)
    duplicate_count = spark.sql(
        f"SELECT COUNT(*) AS duplicate_groups FROM "
        f"(SELECT {key_sql}, COUNT(*) AS n FROM {qualified} "
        f"GROUP BY {key_sql} HAVING COUNT(*) > 1)"
    ).first()["duplicate_groups"]
    assert_condition("T2", duplicate_count == 0, f"{name}: grupos duplicados = {duplicate_count}")
for name in ["steam_games_rejected", "steam_game_reviews_rejected", "steam_player_counts_monthly_rejected", "twitch_game_activity_monthly_rejected"]:
    qualified = table(schemas["dlq"], name)
    duplicate_count = spark.sql(
        f"SELECT COUNT(*) AS duplicate_groups FROM "
        f"(SELECT _reject_key, COUNT(*) AS n FROM {qualified} "
        "GROUP BY _reject_key HAVING COUNT(*) > 1)"
    ).first()["duplicate_groups"]
    assert_condition("T3", duplicate_count == 0, f"dlq.{name}: _reject_key duplicados = {duplicate_count}")


## T4. Historial de escritura

Se imprime el mapa real de métricas de Delta para no asumir nombres de claves que pueden variar según el Runtime. La implementación actual usa reconstrucción completa, por lo que la operación esperada es `CREATE OR REPLACE TABLE` o `WRITE`, no `MERGE`.

In [0]:
for name, qualified in target_tables:
    history = spark.sql(f"DESCRIBE HISTORY {qualified}").orderBy("version", ascending=False).limit(3).collect()
    print(name)
    for row in history:
        print({"version": row["version"], "operation": row["operation"], "operationMetrics": row["operationMetrics"]})
print("PASS T4: historial inspeccionado; las métricas reales fueron impresas sin asumir nombres.")


## T5. Mutación opcional

No se ejecuta por defecto. Si `run_mutation_tests=true`, se modifica únicamente el Bronze sandbox, se vuelve a ejecutar el flujo y se comprueba el cambio válido y dos rechazos DLQ. No se toca Bronze real.

In [0]:
if run_mutation_tests:
    raise NotImplementedError(
        "La mutación requiere seleccionar claves reales del sandbox y confirmar el esquema efectivo antes de ejecutarse."
    )
print("SKIPPED T5: run_mutation_tests=false; no se modificó ningún dato.")


## Limpieza opcional

Elimina el catálogo sandbox completo cuando `cleanup_sandbox=true`.

In [0]:
if cleanup_sandbox:
    spark.sql(f"DROP CATALOG IF EXISTS `{sandbox_catalog}` CASCADE")
    print(f"Catálogo sandbox eliminado: {sandbox_catalog}")
else:
    print(f"Sandbox conservado para inspección: {sandbox_catalog}.{schemas['silver']}, {sandbox_catalog}.{schemas['dlq']}, {sandbox_catalog}.{schemas['gold']}")
